# MNLP 2026 - Homework 2
Serra 1893530

Falanga 1918515

## 1. Setup & Environment
Import libraries, mount storage, and configure global seeds and paths.

In [ ]:
import os
import torch
import pandas as pd
import numpy as np
import json
import pickle
import nltk
import spacy
import datasets
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import time
import random
import ipywidgets as widgets
from IPython.display import display, clear_output
from tqdm.auto import tqdm
from datasets import load_dataset
from google.colab import drive
from sentence_transformers import SentenceTransformer, util
from transformers import AutoTokenizer, AutoModelForCausalLM
from nltk.translate.meteor_score import meteor_score
from sklearn.metrics import cohen_kappa_score

# --- Configuration & Device Setup ---
drive.mount("/content/drive")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED = 42
torch.manual_seed(SEED)
if device.type == 'cuda':
    torch.cuda.manual_seed(SEED)

# --- MODE SWITCH ---
RUN_B1 = False

GROUP = "galline_vecchie_fan_buon_brothers"
TOP_K = 3
BATCH_SIZE = 32
OUT_DIR = "/content/drive/MyDrive/MNLP_HW2_SERRA_FALANGA/outputs"
RETRIEVER_PATH = "/content/drive/MyDrive/MNLP_HW2_SERRA_FALANGA/minilm_ft_weights.pth"
CHECKPOINT_FILE = os.path.join(OUT_DIR, "checkpoint_b1_multi.pkl")
os.makedirs(OUT_DIR, exist_ok=True)

# B2 REQUIREMENT: At least 200 samples for Judge/Manual
N_LLM_SAMPLES = 200

# --- Global Metrics Setup ---
nltk.download('wordnet', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# --- Data Loading ---
ds = load_dataset("sapienzanlp-course-materials/hw-mnlp-2026")
test_ds  = ds['test']
blind_ds = ds['blind']

SMALL_LMS = {
    "qwen3_0_6b": "Qwen/Qwen3-0.6B",
    "smollm2_360m": "HuggingFaceTB/SmolLM2-360M-Instruct",
}
JUDGE_ID = "HuggingFaceTB/SmolLM2-360M-Instruct"

print(f"Setup complete. N_LLM_SAMPLES set to {N_LLM_SAMPLES} as per B2 requirements.")

# --- A3 CACHE CONFIGURATION ---
FORCE_REFETCH_WIKIDATA = False
WIKIDATA_CACHE_PATH = os.path.join(OUT_DIR, "wikidata_cache.json")

## 2. Baseline 1: Inference
Retrieval and generation for Baseline, RAG, and Oracle configurations.

### 2.1 Retriever
Initializing the fine-tuned MiniLM Bi-Encoder for context retrieval.

In [ ]:
# --- Retriever Initialization ---
base_model_name = 'sentence-transformers/all-MiniLM-L6-v2'
retriever = SentenceTransformer(base_model_name, device=device)

print(f"Loading custom weights from {RETRIEVER_PATH}...")
weights = torch.load(RETRIEVER_PATH, map_location=device)
retriever[0].auto_model.load_state_dict(weights, strict=False)

def retrieve_topk_indices(sample, top_k=3):
    query = sample["query"]
    chunks = sample["candidate_chunks"]

    q_emb = retriever.encode(query, convert_to_tensor=True, normalize_embeddings=True)
    c_emb = retriever.encode(chunks, convert_to_tensor=True, normalize_embeddings=True)

    scores = util.cos_sim(q_emb, c_emb)[0]
    topk = torch.topk(scores, k=min(top_k, len(chunks))).indices.cpu().tolist()
    return topk

print("Retriever loaded successfully with fine-tuned weights.")

### 2.2 Language Models
Loading Small LMs for inference.

In [ ]:
def load_lm(model_id):
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto",
        trust_remote_code=True
    )
    # For batching, padding must be on the left to avoid corrupting the generation
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    return tokenizer, model

### 2.3 Prompting Logic
Defining templates for zero-shot and augmented generation.

In [ ]:
def make_baseline_prompt(query):
    return f"""Reply to the question with only the short factual answer.
Do not explain. Do not reason step by step.

Question: {query}
Answer:"""

def make_augmented_prompt(query, chunks):
    context = "\n".join([f"{i+1}. {chunk}" for i, chunk in enumerate(chunks)])
    return f"""Given the following information:

{context}

Reply to the question with only the short factual answer.
Do not explain. Do not reason step by step.

Question: {query}
Answer:"""

def make_oracle_indices(sample, retrieved_indices, top_k=3):
    correct_idx = sample["answer_pos"]
    remaining = [idx for idx in retrieved_indices if idx != correct_idx]
    oracle_indices = [correct_idx] + remaining
    return oracle_indices[:top_k]

### 2.4 Batch Inference
Implementation of the generation loop across data splits.

In [ ]:
def clean_generated_answer(text):
    text = text.strip()

    # Remove Qwen3 thinking block if present
    if "</think>" in text:
        text = text.split("</think>", 1)[1].strip()
    elif text.startswith("<think>"):
        text = text.replace("<think>", "", 1).strip()

    # Remove possible answer prefixes
    for prefix in ["Answer:", "Final answer:", "Final Answer:"]:
        if text.startswith(prefix):
            text = text[len(prefix):].strip()

    # Keep only the first non-empty line to encourage short-answer format
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    return lines[0] if lines else ""


@torch.inference_mode()
def generate_batch_answers(prompts, tokenizer, model, max_new_tokens=48):
    chat_formatted_prompts = [[{"role": "user", "content": p}] for p in prompts]

    try:
        inputs = tokenizer.apply_chat_template(
            chat_formatted_prompts,
            return_tensors="pt",
            return_dict=True,
            padding=True,
            truncation=True,
            max_length=2048,
            add_generation_prompt=True,
            enable_thinking=False
        ).to(model.device)
    except TypeError:
        inputs = tokenizer.apply_chat_template(
            chat_formatted_prompts,
            return_tensors="pt",
            return_dict=True,
            padding=True,
            truncation=True,
            max_length=2048,
            add_generation_prompt=True
        ).to(model.device)

    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    generated_ids = output_ids[:, inputs["input_ids"].shape[-1]:]
    decoded_outputs = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)

    return [clean_generated_answer(text) for text in decoded_outputs]


def save_rag_results(results_list, split_name, short_name, group_name, out_dir):
    if split_name == "test":
        model_filename = f"{group_name}-all-test-{short_name}-rag.jsonl"
    elif split_name == "blind":
        model_filename = f"{group_name}-blind-{short_name}-rag.jsonl"
    else:
        model_filename = f"{group_name}-{split_name}-{short_name}-rag.jsonl"

    save_path = os.path.join(out_dir, model_filename)

    with open(save_path, "w", encoding="utf-8") as f:
        for entry in results_list:
            out_record = {
                "query_id": entry["query_id"],
                "retrieved_chunks": entry["retrieved_chunks"],
                "augmented_prompt": entry["augmented_prompt"],
                "generated_answer": entry["generated_answer"]
            }
            json.dump(out_record, f, ensure_ascii=False)
            f.write("\n")

    print(f"Exported RAG results for {short_name} on {split_name} to {save_path}")


def run_b1_split(data, split_name, model_name, tokenizer, model, include_oracle=True):
    results = {"baseline": [], "rag": [], "oracle": []}

    for i in tqdm(range(0, len(data), BATCH_SIZE), desc=f"{split_name} - {model_name}"):
        batch = data.select(range(i, min(i + BATCH_SIZE, len(data))))
        batch_retrieved_indices = [retrieve_topk_indices(s, TOP_K) for s in batch]

        # Baseline
        b_prompts = [make_baseline_prompt(s["query"]) for s in batch]
        b_ans = generate_batch_answers(b_prompts, tokenizer, model)

        for s, prompt, ans in zip(batch, b_prompts, b_ans):
            results["baseline"].append({
                "query_id": s["query_id"],
                "augmented_prompt": prompt,
                "retrieved_chunks": [],
                "generated_answer": ans
            })

        # RAG
        r_prompts = []
        for s, idxs in zip(batch, batch_retrieved_indices):
            chunks = [s["candidate_chunks"][idx] for idx in idxs]
            r_prompts.append(make_augmented_prompt(s["query"], chunks))

        r_ans = generate_batch_answers(r_prompts, tokenizer, model)

        for s, idxs, prompt, ans in zip(batch, batch_retrieved_indices, r_prompts, r_ans):
            results["rag"].append({
                "query_id": s["query_id"],
                "retrieved_chunks": idxs,
                "augmented_prompt": prompt,
                "generated_answer": ans
            })

        # Oracle
        if include_oracle and "answer_pos" in batch.features:
            oracle_indices_batch = [
                make_oracle_indices(s, idxs, TOP_K)
                for s, idxs in zip(batch, batch_retrieved_indices)
            ]

            o_prompts = []
            for s, oracle_idxs in zip(batch, oracle_indices_batch):
                chunks = [s["candidate_chunks"][idx] for idx in oracle_idxs]
                o_prompts.append(make_augmented_prompt(s["query"], chunks))

            o_ans = generate_batch_answers(o_prompts, tokenizer, model)

            for s, oracle_idxs, prompt, ans in zip(batch, oracle_indices_batch, o_prompts, o_ans):
                results["oracle"].append({
                    "query_id": s["query_id"],
                    "retrieved_chunks": oracle_idxs,
                    "augmented_prompt": prompt,
                    "generated_answer": ans
                })

    return results

### 2.5 Execution
Running inference and saving results.

In [ ]:
if RUN_B1:
    all_results_data = {}

    for short_name, hf_id in SMALL_LMS.items():
        print(f"\nProcessing: {hf_id}")
        tokenizer, model = load_lm(hf_id)

        test_res = run_b1_split(
            test_ds,
            split_name="test",
            model_name=short_name,
            tokenizer=tokenizer,
            model=model,
            include_oracle=True
        )

        blind_res = run_b1_split(
            blind_ds,
            split_name="blind",
            model_name=short_name,
            tokenizer=tokenizer,
            model=model,
            include_oracle=False
        )

        all_results_data[short_name] = {
            "test": test_res,
            "blind": blind_res
        }

        # Save only RAG outputs as required by the slides
        save_rag_results(test_res["rag"], "test", short_name, GROUP, OUT_DIR)
        save_rag_results(blind_res["rag"], "blind", short_name, GROUP, OUT_DIR)

        # Save checkpoint after each model
        with open(CHECKPOINT_FILE, "wb") as f:
            pickle.dump({"all_results_data": all_results_data}, f)

        del model, tokenizer
        torch.cuda.empty_cache()

else:
    print(f"Loading results from {CHECKPOINT_FILE}...")
    with open(CHECKPOINT_FILE, "rb") as f:
        all_results_data = pickle.load(f)["all_results_data"]
    print(f"Models loaded from disk: {list(all_results_data.keys())}")

## 3. Baseline 2: Evaluation
Automated metrics, LLM-as-a-Judge, and human validation agreement.

### 3.1 Automated Metrics
Computing EM, subEM, and METEOR across all systems.

In [ ]:
def calculate_em(preds, targets_list):
    # 1 if any valid target matches the prediction exactly
    scores = [1 if any(p.strip().lower() == t.strip().lower() for t in ts) else 0 for p, ts in zip(preds, targets_list)]
    return np.mean(scores) * 100

def calculate_sub_em(preds, targets_list):
    # 1 if any valid target is a substring of the prediction
    scores = [1 if any(t.strip().lower() in p.strip().lower() for t in ts) else 0 for p, ts in zip(preds, targets_list)]
    return np.mean(scores) * 100

def calculate_meteor(preds, targets_list):
    scores = []
    for p, ts in zip(preds, targets_list):
        p_tokens = nltk.word_tokenize(p.strip())
        ts_tokens = [nltk.word_tokenize(t.strip()) for t in ts]
        scores.append(meteor_score(ts_tokens, p_tokens))
    return np.mean(scores) * 100

def run_automated_evaluation(results_dict, dataset):
    id_to_short = {s['query_id']: s['short_answer'] for s in dataset}
    report = []
    for cfg in ['baseline', 'rag', 'oracle']:
        data = results_dict.get(cfg, [])
        if not data: continue
        preds = [item['generated_answer'] for item in data]
        targets = [id_to_short[item['query_id']] for item in data]
        report.append({
            "Config": cfg.upper(),
            "EM": calculate_em(preds, targets),
            "subEM": calculate_sub_em(preds, targets),
            "METEOR": calculate_meteor(preds, targets)
        })
    return pd.DataFrame(report)

In [ ]:
all_reports = {}
for m_name, res in all_results_data.items():
    print(f"\n--- Metrics for {m_name} ---")
    report = run_automated_evaluation(res['test'], test_ds)
    all_reports[m_name] = report
    display(report)

# B2 Requirement: Select best RAG system
best_model_name = max(all_reports, key=lambda m: all_reports[m].loc[all_reports[m]['Config'] == 'RAG', 'METEOR'].values[0])
best_rag_results = all_results_data[best_model_name]['test']['rag']
print(f"\nSelected Best System for Judge/Manual: {best_model_name}")

# --- Visualization & Save to Drive ---
plot_data = []
for m_name, report in all_reports.items():
    m_df = report.copy()
    m_df['Model'] = m_name
    plot_data.append(m_df)

df_plot = pd.concat(plot_data).melt(id_vars=['Config', 'Model'], var_name='Metric', value_name='Percentage')

plt.figure(figsize=(14, 7))
sns.set_theme(style="whitegrid")
g = sns.catplot(
    data=df_plot, kind="bar",
    x="Config", y="Percentage", hue="Model", col="Metric",
    palette="magma", height=5, aspect=0.8
)
g.set_axis_labels("Configuration", "Percentage")
g.set_titles("{col_name}")

# Save image to Drive
save_path = os.path.join(OUT_DIR, "model_comparison.png")
plt.savefig(save_path, dpi=300, bbox_inches='tight')
print(f"\nComparison plot saved to: {save_path}")
plt.show()

### 3.2 LLM-as-a-Judge
Binary evaluation of 200 samples from the best-performing RAG system.

In [ ]:
judge_tokenizer, judge_model = load_lm(JUDGE_ID)

def make_llm_eval_prompt(query, ground_truth, generated_answer):
    return f"""Evaluate whether the generated answer contains the correct short answer.

Use this criterion:
- 1 if the correct short answer is present in any valid form.
- 0 if the correct short answer is not present.

Output only 1 or 0.

Query: {query}
Correct short answer: {ground_truth}
Generated answer: {generated_answer}

Score:"""

np.random.seed(SEED)

id_to_sample = {s["query_id"]: s for s in test_ds}
all_test_ids = [s["query_id"] for s in test_ds]

sample_ids = np.random.choice(
    all_test_ids,
    size=N_LLM_SAMPLES,
    replace=False
)

best_rag_map = {
    item["query_id"]: item
    for item in best_rag_results
}

llm_judge_scores = []

for qid in tqdm(sample_ids, desc="LLM Judging"):
    s = id_to_sample[qid]
    generated_answer = best_rag_map[qid]["generated_answer"]

    prompt = make_llm_eval_prompt(
        query=s["query"],
        ground_truth=s["short_answer"],
        generated_answer=generated_answer
    )

    messages = [[{"role": "user", "content": prompt}]]

    inputs = judge_tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        return_dict=True,
        padding=True,
        truncation=True,
        max_length=2048,
        add_generation_prompt=True
    ).to(judge_model.device)

    with torch.no_grad():
        output = judge_model.generate(
            **inputs,
            max_new_tokens=2,
            do_sample=False,
            pad_token_id=judge_tokenizer.pad_token_id,
            eos_token_id=judge_tokenizer.eos_token_id
        )

    prompt_length = inputs["input_ids"].shape[1]
    res = judge_tokenizer.decode(
        output[0, prompt_length:],
        skip_special_tokens=True
    ).strip()

    llm_judge_scores.append(1 if "1" in res else 0)

print(
    f"Judge Evaluation Complete on {len(llm_judge_scores)} samples. "
    f"Judge positive rate: {np.mean(llm_judge_scores) * 100:.2f}%"
)

### 3.3 Manual Validation & Agreement
Calculating Cohen's Kappa between annotators and the LLM Judge.

In [ ]:
# --- SHARED PERSISTENT MANUAL EVALUATION (Antonio & Daniele) ---
MANUAL_SCORES_PATH = os.path.join(OUT_DIR, "manual_scores_progress.json")

def save_manual_progress():
    progress = {
        "annotator_1": annotator_1_scores,
        "annotator_2": annotator_2_scores
    }
    with open(MANUAL_SCORES_PATH, "w") as f:
        json.dump(progress, f)

def load_manual_progress():
    if os.path.exists(MANUAL_SCORES_PATH):
        with open(MANUAL_SCORES_PATH, "r") as f:
            data = json.load(f)
            a1 = data.get("annotator_1", [None]*N_LLM_SAMPLES)
            a2 = data.get("annotator_2", [None]*N_LLM_SAMPLES)
            return a1, a2
    return [None]*N_LLM_SAMPLES, [None]*N_LLM_SAMPLES

# Initialization
annotator_1_scores, annotator_2_scores = load_manual_progress()
current_idx = 0
out = widgets.Output()

def update_view():
    with out:
        clear_output(wait=True)
        qid = sample_ids[current_idx]
        sample = id_to_sample[qid]
        rag_ans = best_rag_map[qid]['generated_answer']
        v1_val, v2_val = annotator_1_scores[current_idx], annotator_2_scores[current_idx]

        status = "🟢 Completed" if (v1_val is not None and v2_val is not None) else ("🟡 In Progress" if (v1_val is not None or v2_val is not None) else "🔴 Pending")

        print(f"QID {qid}")
        print(f"SAMPLE {current_idx + 1} / {N_LLM_SAMPLES} | STATUS: {status}")
        print(f"\nQUERY: {sample['query']}")
        print(f"GT: {' | '.join(sample['short_answer'])}")
        print(f"RAG: {rag_ans}")
        print(f"\n--- VOTI ATTUALI ---")
        print(f"Antonio: {'✅' if v1_val==1 else ('❌' if v1_val==0 else '...')}", end=" | ")
        print(f"Daniele: {'✅' if v2_val==1 else ('❌' if v2_val==0 else '...')}")

def set_score(ann_num, score):
    if ann_num == 1: annotator_1_scores[current_idx] = score
    else: annotator_2_scores[current_idx] = score
    save_manual_progress()
    update_view()

def move(step):
    global current_idx
    current_idx = max(0, min(N_LLM_SAMPLES - 1, current_idx + step))
    update_view()

# UI Setup
btn_prev = widgets.Button(description="⬅️ Previous"); btn_prev.on_click(lambda b: move(-1))
btn_next = widgets.Button(description="Next ➡️"); btn_next.on_click(lambda b: move(1))
btn_a1_0 = widgets.Button(description="Antonio: Wrong", button_style='danger'); btn_a1_0.on_click(lambda b: set_score(1, 0))
btn_a1_1 = widgets.Button(description="Antonio: Correct", button_style='success'); btn_a1_1.on_click(lambda b: set_score(1, 1))
btn_a2_0 = widgets.Button(description="Daniele: Wrong", button_style='danger'); btn_a2_0.on_click(lambda b: set_score(2, 0))
btn_a2_1 = widgets.Button(description="Daniele: Correct", button_style='success'); btn_a2_1.on_click(lambda b: set_score(2, 1))

display(widgets.VBox([
    widgets.HBox([btn_prev, btn_next]),
    widgets.HBox([widgets.Label("Antonio:"), btn_a1_0, btn_a1_1]),
    widgets.HBox([widgets.Label("Daniele:"), btn_a2_0, btn_a2_1]),
    out
]))
update_view()

In [ ]:
if len(annotator_1_scores) == len(llm_judge_scores) and len(annotator_2_scores) == len(llm_judge_scores):
    # 1. Calculate Automated Metrics on the 200 samples
    sample_rag_data = [best_rag_map[qid] for qid in sample_ids]
    subset_preds = [item['generated_answer'] for item in sample_rag_data]
    subset_targets = [id_to_sample[qid]['short_answer'] for qid in sample_ids]

    subset_em = calculate_em(subset_preds, subset_targets)
    subset_sub_em = calculate_sub_em(subset_preds, subset_targets)
    subset_meteor = calculate_meteor(subset_preds, subset_targets)

    # 2. Calculate Cohen's Kappa
    k_h1_h2 = cohen_kappa_score(annotator_1_scores, annotator_2_scores)
    k_h1_lj = cohen_kappa_score(annotator_1_scores, llm_judge_scores)
    k_h2_lj = cohen_kappa_score(annotator_2_scores, llm_judge_scores)

    # 3. Print All Results
    print(f"--- EVALUATION SUMMARY (Subset of {N_LLM_SAMPLES} samples) ---")
    print(f"System: {best_model_name}\n")

    print("Automated Metrics:")
    print(f"- EM: {subset_em:.2f}")
    print(f"- subEM: {subset_sub_em:.2f}")
    print(f"- METEOR: {subset_meteor:.2f}\n")

    print("Agreement Metrics (Cohen's Kappa):")
    print(f"- Human1 vs Human2: {k_h1_h2:.4f}")
    print(f"- Human1 vs LLM Judge: {k_h1_lj:.4f}")
    print(f"- Human2 vs LLM Judge: {k_h2_lj:.4f}\n")

    print("Judge Performance:")
    print(f"- LLM Judge Positive Rate: {np.mean(llm_judge_scores) * 100:.2f}%\n")

    # 4. Save results to disk
    os.makedirs(OUT_DIR, exist_ok=True)
    output_file = os.path.join(
        OUT_DIR,
        f"{GROUP}-judge-subset-{best_model_name}-rag.jsonl"
    )

    with open(output_file, "w", encoding="utf-8") as f:
        for i, qid in enumerate(sample_ids):
            rag_entry = best_rag_map[qid]
            sample = id_to_sample[qid]

            entry = {
                "query_id": qid,
                "retrieved_chunks": rag_entry["retrieved_chunks"],
                "augmented_prompt": rag_entry["augmented_prompt"],
                "ground_truth": sample["short_answer"],
                "generated_answer": rag_entry["generated_answer"],
                "llm_judge_a": llm_judge_scores[i],
                "annotator_1": annotator_1_scores[i],
                "annotator_2": annotator_2_scores[i]
            }
            json.dump(entry, f, ensure_ascii=False)
            f.write("\n")

    print(f"Evaluation results successfully saved to {output_file}")

else:
    print("Error: Length mismatch between judge scores and manual scores.")

### 3.4 Qualitative analysis of limitations and weaknesses

Based on the manually annotated subset of 200 samples, use the following tool to identify discrepancies and qualitative patterns.

In [ ]:
# --- FAILURE MODE ANALYSIS TOOL ---
# This cell helps identify samples where the system or the judge failed.

disagreements = []
for i, qid in enumerate(sample_ids):
    # Fix: Reference best_rag_map[qid]['generated_answer']
    rag_ans = best_rag_map[qid]['generated_answer']

    # Condition: judge differs from human or human marked as incorrect
    if annotator_1_scores[i] != llm_judge_scores[i] or annotator_1_scores[i] == 0:
        disagreements.append({
            'index': i,
            'query_id': qid,
            'query': id_to_sample[qid]['query'],
            'gt': " | ".join(id_to_sample[qid]['short_answer']),
            'rag_ans': rag_ans,
            'h1': annotator_1_scores[i],
            'judge': llm_judge_scores[i]
        })

print(f"Found {len(disagreements)} potential failure/disagreement cases.\n")

# Display the first 5 cases for inspection
for case in disagreements[:5]:
    print(f"--- Sample Index {case['index']} ---")
    print(f"Q: {case['query']}")
    print(f"GT: {case['gt']}")
    print(f"Model Output: {case['rag_ans']}")
    print(f"Scores -> Human: {case['h1']}, Judge: {case['judge']}\n")

### 3.5 Numerical Analysis of Results

#### 1. Model Comparison Table
The following table summarizes the performance of the two tested models across different configurations on the full test set:

| Model | Config | EM (%) | subEM (%) | METEOR (%) |
| :--- | :--- | :--- | :--- | :--- |
| **Qwen-0.6B** | BASELINE | 0.30 | 2.15 | 5.41 |
| | RAG | 5.65 | 27.60 | 30.63 |
| | ORACLE | 6.70 | 33.75 | 34.33 |
| **SmolLM2-360M** | BASELINE | 0.75 | 6.15 | 8.96 |
| | **RAG** | **15.85** | **32.25** | **32.88** |
| | ORACLE | 16.90 | 34.05 | 34.62 |

#### 2. Key Considerations
*   **SmolLM2 Superiority:** Despite being smaller in parameter count, `SmolLM2-360M` significantly outperforms `Qwen-0.6B`, especially in the RAG configuration where it more than doubles the EM score (15.85% vs 5.65%).
*   **Retrieval Impact:** The jump from BASELINE to RAG is transformative. For SmolLM2, the subEM improves by over 26 percentage points, confirming that these models rely heavily on the provided context to bridge their internal knowledge gaps.
*   **Retrieval Efficiency:** The RAG performance is remarkably close to the ORACLE performance (within ~2% for METEOR). This indicates that our fine-tuned MiniLM retriever is highly effective at identifying the ground-truth context.

#### 3. Human Validation and Judge Reliability
*   **Human Agreement:** Near-perfect alignment between annotators (**Kappa 0.9898**).
*   **Judge Bias:** The LLM Judge (SmolLM2) is overly lenient, with a **96.5% positive rate** and a very low agreement with humans (**Kappa ~0.07**). This highlights that while SmolLM2 is a good generator, it struggles as a reliable evaluator for factual correctness in this specific domain.

## Advanced Task A3: Wikidata Integration
In this section, we enhance our RAG pipeline by integrating external knowledge from Wikidata. We fetch entity descriptions to provide the model with factual grounding beyond the retrieved document chunks.

### A3.1 Metadata Acquisition
We use the Wikidata API to fetch labels and descriptions. To comply with Wikimedia's policy, we implement:
1.  **Custom User-Agent headers** identifying our project.
2.  **Rate limiting** (1-second delay between requests).
3.  **Exponential backoff** to gracefully handle `429 Too Many Requests` errors.

In [ ]:
def load_wikidata_cache():
    if os.path.exists(WIKIDATA_CACHE_PATH) and not FORCE_REFETCH_WIKIDATA:
        print(f"Loading Wikidata cache from {WIKIDATA_CACHE_PATH}...")
        with open(WIKIDATA_CACHE_PATH, 'r', encoding='utf-8') as f:
            return json.load(f)
    return {}

def save_wikidata_cache(cache):
    print(f"Saving Wikidata cache to {WIKIDATA_CACHE_PATH}...")
    with open(WIKIDATA_CACHE_PATH, 'w', encoding='utf-8') as f:
        json.dump(cache, f, ensure_ascii=False, indent=2)

# Load existing data if available
wikidata_cache = load_wikidata_cache()

In [ ]:
def fetch_wikidata_metadata(entity_id, max_retries=10):
    """Fetches label and description from Wikidata API following Wikimedia guidelines."""
    if not entity_id or pd.isna(entity_id):
        return None, None

    url = "https://www.wikidata.org/w/api.php"
    headers = {
        'User-Agent': f'MNLP-HW2-Client/1.0 ({GROUP}; bot-traffic@wikimedia.org)',
        'Api-User-Agent': f'MNLP-HW2-Client/1.0 ({GROUP})'
    }
    params = {
        "action": "wbgetentities",
        "ids": entity_id,
        "languages": "en",
        "props": "labels|descriptions",
        "format": "json"
    }

    for attempt in range(max_retries):
        try:
            response = requests.get(url, params=params, headers=headers, timeout=15)
            if response.status_code == 200:
                data = response.json().get('entities', {}).get(entity_id, {})
                label = data.get('labels', {}).get('en', {}).get('value')
                description = data.get('descriptions', {}).get('en', {}).get('value')
                return label, description
            elif response.status_code == 429:
                wait_time = 2 ** (attempt + 1)
                print(f"[429] Rate limit exceeded. Waiting for {wait_time} seconds...")
                time.sleep(wait_time)
            else:
                print(f"Error {response.status_code}: {response.text}")
                break
        except Exception:
            print(f"[Exception] Request failed. Retrying in 1 second...")
            time.sleep(1)
    return None, None

# Check if we should fetch or use cache
if FORCE_REFETCH_WIKIDATA:
    print("A3.1: Force refetch enabled. Clearing local cache...")
    wikidata_cache = {}

print(f"A3.1: Fetching missing Wikidata metadata for test set...")
for s in tqdm(test_ds):
    w_id = s.get('wikidata_id')
    if w_id and w_id not in wikidata_cache:
        label, desc = fetch_wikidata_metadata(w_id)
        wikidata_cache[w_id] = (label, desc)
        time.sleep(1.0) # Respect rate limits

print(f"Acquisition complete. Items in cache: {len(wikidata_cache)}")

In [ ]:
# Save the updated cache after the fetching loop completes
save_wikidata_cache(wikidata_cache)

In [ ]:
# Identify and print incomplete Wikidata entries (null labels or descriptions)
incomplete_entries = []

for qid, (label, desc) in wikidata_cache.items():
    if label is None or desc is None:
        incomplete_entries.append({"id": qid, "label": label, "description": desc})

print(f"Found {len(incomplete_entries)} incomplete entries in wikidata_cache.\n")
for entry in incomplete_entries[:10]: # Displaying first 10 for brevity
    print(entry)

if len(incomplete_entries) > 10:
    print("...")

### A3.2 Wikidata-Enriched RAG Inference
We augment the prompt by injecting the Wikidata description as "Additional Knowledge" at the top of the context. This helps the model reconcile information between the retrieved chunks and structured knowledge.

In [ ]:
def make_enriched_prompt(query, chunks, wikidata_desc=None):
    """Creates prompt with Wikidata info at the top, emphasizing short answers."""
    context = "\n".join([f"{i+1}. {c}" for i, c in enumerate(chunks)])
    extra = f"Additional Knowledge (Wikidata): {wikidata_desc}\n" if wikidata_desc else ""

    return f"""{extra}
      Given the information above and the context below:
      {context}

      Reply to the question with only the short factual answer.
      Do not explain. Do not reason step by step.

      Question: {query}
      Answer:"""

print(f"A3.2: Running Enriched RAG Inference (Full Test Set)...")
tokenizer, model = load_lm(SMALL_LMS[best_model_name])
enriched_answers = []

# Prepare all prompts first
enriched_prompts = []
for s in test_ds:
    qid = s['query_id']
    _, desc = wikidata_cache.get(s.get('wikidata_id'), (None, None))
    retrieved_indices = best_rag_map[qid]['retrieved_chunks']
    chunks = [s['candidate_chunks'][i] for i in retrieved_indices]
    enriched_prompts.append(make_enriched_prompt(s['query'], chunks, desc))

# Batch generation
for i in tqdm(range(0, len(enriched_prompts), BATCH_SIZE)):
    batch = enriched_prompts[i : i + BATCH_SIZE]
    batch_ans = generate_batch_answers(batch, tokenizer, model)
    enriched_answers.extend(batch_ans)

del model, tokenizer
torch.cuda.empty_cache()

### A3.3 Metrics & Evaluation
We compare the performance of the **Standard RAG** versus the **Wikidata-Enriched RAG** using automated metrics (EM, subEM, and METEOR) across the full test set.

In [ ]:
# A3.3: Metrics Calculation & Comparison
full_targets = [s['short_answer'] for s in test_ds]
std_preds = [best_rag_map[s['query_id']]['generated_answer'] for s in test_ds]

comparison_data = []

# Standard RAG Metrics
comparison_data.append({
    "Config": "Standard RAG",
    "EM": calculate_em(std_preds, full_targets),
    "subEM": calculate_sub_em(std_preds, full_targets),
    "METEOR": calculate_meteor(std_preds, full_targets)
})

# Wikidata-Enriched RAG Metrics
comparison_data.append({
    "Config": "Wikidata-Enriched RAG",
    "EM": calculate_em(enriched_answers, full_targets),
    "subEM": calculate_sub_em(enriched_answers, full_targets),
    "METEOR": calculate_meteor(enriched_answers, full_targets)
})

final_report = pd.DataFrame(comparison_data)
print("--- A3.3: Final Performance Comparison (Full Test Set) ---")
display(final_report)

### A3.4 Final Comprehensive Analysis & Wikidata Impact

#### 1. Final Performance Summary
The table below summarizes the transition from a standard RAG pipeline to one enriched with Wikidata metadata across the full test set for our best-performing model (`SmolLM2-360M`).

| Configuration | EM (%) | subEM (%) | METEOR (%) |
| :--- | :---: | :---: | :---: |
| **Standard RAG** | **15.85** | 32.25 | 32.88 |
| **Wikidata-Enriched RAG** | 15.75 | **32.70** | **33.06** |

#### 2. Qualitative Findings on Wikidata Integration
The integration of Wikidata descriptions serves as a **semantic anchor** that complements the unstructured information retrieved from Wikipedia chunks. Our analysis reveals three main patterns:

*   **Entity Disambiguation:** In complex queries where multiple entities are mentioned in the context (e.g., distinguishing between a creator and their work), the Wikidata description provided at the top of the prompt acts as a "system hint" that keeps the model focused on the primary subject. This is evidenced by the improvement in **subEM (+0.45%)**, suggesting that while the exact wording might vary, the model captures the core factual answer more frequently.
*   **Contextual Boosting:** We observed cases where the retriever failed to return the exact sentence containing the answer, but the Wikidata description provided enough categorical information (e.g., "American actor" or "1917 battle") to allow the model to narrow down the correct answer from the available text.
*   **Stability vs. Noise:** While the performance gain is modest, it is remarkably consistent. However, the slight dip in Exact Match (EM) indicates that adding more text to the prompt can sometimes lead to slightly more verbose answers, which penalizes string-literal metrics even when the factual correctness is preserved.

#### 3. Overall Project Conclusions
*   **The Power of Small Models:** This project demonstrates that a 360M parameter model, when supported by a robust RAG pipeline and external knowledge bases, can achieve performance levels close to its theoretical ceiling (ORACLE), bridging the gap created by its limited internal parameters.
*   **The Judge Paradox:** One of the most striking findings is the failure of the **LLM-as-a-Judge**. The high leniency and low agreement with humans (**Kappa ~0.07**) suggest that model-based evaluation requires significantly larger or more specialized models to be reliable.
*   **Future Directions:** Improving the RAG pipeline further would likely require more advanced metadata fetching (e.g., fetching Wikidata relations, not just descriptions) or implementing a re-ranking stage to prioritize the high-quality chunks before feeding them to the LLM.